In [14]:
import warnings
warnings.filterwarnings("ignore")

In [1]:
import pandas as pd
fp = "Depression Professional Dataset.csv"

f:\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv(fp)
feature_df=df.drop('Depression',axis=1)
labels = df['Depression'].map({'Yes':1,"No":0}).astype(int)

In [3]:
feature_df = pd.get_dummies(feature_df)
feature_df = feature_df.astype(float)

In [4]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
f_s = sc.fit_transform(feature_df)
feature_df = pd.DataFrame(f_s,columns=sc.get_feature_names_out(feature_df.columns))

In [5]:
sc.get_feature_names_out(feature_df.columns)

array(['Age', 'Work Pressure', 'Job Satisfaction', 'Work Hours',
       'Financial Stress', 'Gender_Female', 'Gender_Male',
       'Sleep Duration_5-6 hours', 'Sleep Duration_7-8 hours',
       'Sleep Duration_Less than 5 hours',
       'Sleep Duration_More than 8 hours', 'Dietary Habits_Healthy',
       'Dietary Habits_Moderate', 'Dietary Habits_Unhealthy',
       'Have you ever had suicidal thoughts ?_No',
       'Have you ever had suicidal thoughts ?_Yes',
       'Family History of Mental Illness_No',
       'Family History of Mental Illness_Yes'], dtype=object)

In [7]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [8]:
train_x,test_x,train_y,test_y = train_test_split(torch.tensor(feature_df.values,dtype=torch.float32),torch.tensor(labels,dtype=torch.int),test_size=0.2)
train_set = TensorDataset(train_x,train_y)
test_set = TensorDataset(test_x,test_y)
bs=64
train_loader = DataLoader(train_set,batch_size=bs)
test_loader = DataLoader(test_set,batch_size=bs)

In [9]:
class FFNN(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim,input_dim//2),nn.LayerNorm(input_dim//2),nn.ReLU(),nn.Linear(input_dim//2,1))
    def forward(self,x):
        return self.net(x)

In [10]:
model=FFNN(train_x.shape[1])
optimizer = AdamW(model.parameters(),lr=2e-3)

In [11]:
lf = nn.BCEWithLogitsLoss()

In [12]:
from sklearn.metrics import accuracy_score, classification_report

In [15]:
n_e = 3
for epoch in range(n_e):
    le = 0.
    model.train()
    for x,y in train_loader:
        pred = model(x)
        loss = lf(pred,y.float().view(-1,1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        le += loss.float()*x.shape[0]
    le /= train_x.shape[0]
    print(f"Epoch:{epoch} Loss: {le:.4f}")
    
    
    model.eval()
    preds= []
    for x,y in test_loader:
        pred = model(x)
        p_label = list((pred.view(-1)>0.5).to(int).numpy())
        preds.extend(p_label)
    print(f"accuracy:{accuracy_score(test_y,preds)}")
    report = classification_report(test_y,preds)
    print(report)

Epoch:0 Loss: 0.2894
accuracy:0.902676399026764
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       371
           1       0.00      0.00      0.00        40

    accuracy                           0.90       411
   macro avg       0.45      0.50      0.47       411
weighted avg       0.81      0.90      0.86       411

Epoch:1 Loss: 0.2405
accuracy:0.902676399026764
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       371
           1       0.00      0.00      0.00        40

    accuracy                           0.90       411
   macro avg       0.45      0.50      0.47       411
weighted avg       0.81      0.90      0.86       411

Epoch:2 Loss: 0.1975
accuracy:0.902676399026764
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       371
           1       0.00      0.00      0.00        40

    accuracy                          